<a href="https://colab.research.google.com/github/Roger-Quinelato/dtLab-2/blob/claude/Quest%C3%A3o_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# SOLUÇÕES PARA MELHORAR A ACURÁCIA DO RECONHECIMENTO FACIAL COM MÁSCARA

# =============================================================================
# SOLUÇÃO 1: MÚLTIPLAS IMAGENS DE REFERÊNCIA (MAIS RECOMENDADA)
# =============================================================================
!pip install deepface

import os
import pickle
import numpy as np
import cv2
from deepface import DeepFace
from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt

def create_enhanced_gallery_multiple_references():
    """
    Cria um banco de dados com múltiplas imagens de referência por pessoa,
    incluindo imagens com diferentes ângulos, iluminação e expressões.
    """

    # Caminhos das múltiplas imagens do Marcelinho
    marcelinho_images = [
        "/content/marcelinho_no_db.jpg",  # Imagem original
        # Adicione mais imagens aqui se disponíveis
        # "/content/marcelinho_perfil.jpg",
        # "/content/marcelinho_sorrindo.jpg",
    ]

    enhanced_gallery = []

    # Carrega galeria de celebridades existente
    if os.path.exists("/content/embedding_gallery_3000.pkl"):
        with open("/content/embedding_gallery_3000.pkl", 'rb') as f:
            celebrity_gallery = pickle.load(f)
        enhanced_gallery.extend(celebrity_gallery)

    # Adiciona múltiplas referências do Marcelinho
    for i, img_path in enumerate(marcelinho_images):
        if os.path.exists(img_path):
            try:
                embedding_obj = DeepFace.represent(img_path, 'ArcFace', enforce_detection=True)
                entry = {
                    "identity": "Marcelinho",
                    "embedding": embedding_obj[0]['embedding'],
                    "image_path": img_path,
                    "reference_id": i  # Para identificar diferentes referências
                }
                enhanced_gallery.append(entry)
                print(f"✅ Adicionada referência {i+1} do Marcelinho: {img_path}")
            except Exception as e:
                print(f"❌ Erro ao processar {img_path}: {e}")

    # Salva galeria aprimorada
    with open("enhanced_gallery.pkl", 'wb') as f:
        pickle.dump(enhanced_gallery, f)

    return enhanced_gallery

# =============================================================================
# SOLUÇÃO 2: SISTEMA DE VOTAÇÃO COM MÚLTIPLAS REFERÊNCIAS
# =============================================================================

def enhanced_face_recognition_with_voting(probe_image_path, gallery, threshold=0.40):
    """
    Sistema de reconhecimento com votação usando múltiplas referências.
    """

    # Gera embedding da imagem de teste
    try:
        probe_embedding_obj = DeepFace.represent(probe_image_path, 'ArcFace', enforce_detection=True)
        probe_embedding = probe_embedding_obj[0]['embedding']
    except Exception as e:
        print(f"Erro ao processar imagem de teste: {e}")
        return None

    # Agrupa resultados por identidade
    identity_scores = {}

    for entry in gallery:
        identity = entry['identity']
        gallery_embedding = entry['embedding']
        distance = cosine(probe_embedding, gallery_embedding)

        if identity not in identity_scores:
            identity_scores[identity] = []

        identity_scores[identity].append({
            'distance': distance,
            'reference_image': entry['image_path']
        })

    # Calcula scores finais por identidade
    final_results = []
    for identity, scores in identity_scores.items():
        # Estratégias de agregação:
        min_distance = min(score['distance'] for score in scores)  # Melhor match
        avg_distance = np.mean([score['distance'] for score in scores])  # Média
        median_distance = np.median([score['distance'] for score in scores])  # Mediana

        # Conta quantas referências estão abaixo do threshold
        valid_matches = sum(1 for score in scores if score['distance'] < threshold)
        confidence = valid_matches / len(scores)

        final_results.append({
            'identity': identity,
            'min_distance': min_distance,
            'avg_distance': avg_distance,
            'median_distance': median_distance,
            'confidence': confidence,
            'num_references': len(scores),
            'best_reference': min(scores, key=lambda x: x['distance'])['reference_image']
        })

    # Ordena por menor distância mínima
    final_results.sort(key=lambda x: x['min_distance'])

    return final_results

# =============================================================================
# SOLUÇÃO 3: PRÉ-PROCESSAMENTO AVANÇADO DE IMAGENS
# =============================================================================

def preprocess_image_for_better_recognition(image_path, output_path=None):
    """
    Aplica técnicas de pré-processamento para melhorar o reconhecimento.
    """

    image = cv2.imread(image_path)

    # 1. Equalização de histograma para melhorar contraste
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    equalized = cv2.equalizeHist(gray)
    image_eq = cv2.cvtColor(equalized, cv2.COLOR_GRAY2BGR)

    # 2. CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_applied = clahe.apply(gray)
    image_clahe = cv2.cvtColor(clahe_applied, cv2.COLOR_GRAY2BGR)

    # 3. Redução de ruído
    denoised = cv2.bilateralFilter(image, 9, 75, 75)

    # 4. Sharpening (realce de bordas)
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(image, -1, kernel)

    if output_path:
        cv2.imwrite(output_path, denoised)  # Salva versão com redução de ruído

    return {
        'original': image,
        'equalized': image_eq,
        'clahe': image_clahe,
        'denoised': denoised,
        'sharpened': sharpened
    }

# =============================================================================
# SOLUÇÃO 4: ENSEMBLE DE MODELOS
# =============================================================================

def ensemble_face_recognition(probe_image_path, gallery):
    """
    Usa múltiplos modelos para melhorar a acurácia.
    """

    models = ['ArcFace', 'Facenet', 'Facenet512', 'OpenFace']
    ensemble_results = {}

    for model_name in models:
        try:
            print(f"Testando modelo: {model_name}")

            # Gera embedding da imagem de teste
            probe_embedding_obj = DeepFace.represent(probe_image_path, model_name, enforce_detection=True)
            probe_embedding = probe_embedding_obj[0]['embedding']

            # Compara com galeria (assumindo que a galeria tem embeddings do mesmo modelo)
            results = []
            for entry in gallery:
                if entry['identity'] == 'Marcelinho':  # Foca apenas no Marcelinho para teste
                    # Re-gera embedding da referência com o modelo atual
                    ref_embedding_obj = DeepFace.represent(entry['image_path'], model_name, enforce_detection=True)
                    ref_embedding = ref_embedding_obj[0]['embedding']

                    distance = cosine(probe_embedding, ref_embedding)
                    results.append({
                        'identity': entry['identity'],
                        'distance': distance,
                        'model': model_name
                    })

            if results:
                ensemble_results[model_name] = min(results, key=lambda x: x['distance'])

        except Exception as e:
            print(f"Erro com modelo {model_name}: {e}")

    return ensemble_results

# =============================================================================
# SOLUÇÃO 5: AJUSTE DINÂMICO DE THRESHOLD
# =============================================================================

def adaptive_threshold_recognition(probe_image_path, gallery):
    """
    Calcula threshold adaptativo baseado na distribuição de distâncias.
    """

    # Gera embedding da imagem de teste
    probe_embedding_obj = DeepFace.represent(probe_image_path, 'ArcFace', enforce_detection=True)
    probe_embedding = probe_embedding_obj[0]['embedding']

    # Calcula todas as distâncias
    all_distances = []
    results = []

    for entry in gallery:
        distance = cosine(probe_embedding, entry['embedding'])
        all_distances.append(distance)
        results.append({
            'identity': entry['identity'],
            'distance': distance,
            'image_path': entry['image_path']
        })

    # Calcula estatísticas
    mean_distance = np.mean(all_distances)
    std_distance = np.std(all_distances)

    # Threshold adaptativo (média - 1 desvio padrão)
    adaptive_threshold = mean_distance - std_distance

    # Encontra melhor match
    best_match = min(results, key=lambda x: x['distance'])

    return {
        'best_match': best_match,
        'adaptive_threshold': adaptive_threshold,
        'mean_distance': mean_distance,
        'std_distance': std_distance,
        'is_valid_match': best_match['distance'] < adaptive_threshold
    }

# =============================================================================
# SOLUÇÃO 6: ANÁLISE DE REGIÃO FACIAL (FOCO NA ÁREA DOS OLHOS)
# =============================================================================

def extract_eye_region_embedding(image_path):
    """
    Extrai embedding focando na região dos olhos (não coberta pela máscara).
    """

    image = cv2.imread(image_path)

    # Detecta faces usando OpenCV
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    if len(faces) > 0:
        (x, y, w, h) = faces[0]

        # Extrai região superior da face (onde estão os olhos)
        eye_region_y_start = y
        eye_region_y_end = y + int(h * 0.6)  # 60% superior da face

        eye_region = image[eye_region_y_start:eye_region_y_end, x:x+w]

        # Salva região dos olhos temporariamente
        temp_path = "/tmp/eye_region.jpg"
        cv2.imwrite(temp_path, eye_region)

        try:
            # Gera embedding da região dos olhos
            embedding_obj = DeepFace.represent(temp_path, 'ArcFace', enforce_detection=False)
            return embedding_obj[0]['embedding']
        except:
            # Se falhar, usa a imagem completa
            embedding_obj = DeepFace.represent(image_path, 'ArcFace', enforce_detection=True)
            return embedding_obj[0]['embedding']

    else:
        # Se não detectar face, usa a imagem completa
        embedding_obj = DeepFace.represent(image_path, 'ArcFace', enforce_detection=True)
        return embedding_obj[0]['embedding']

# =============================================================================
# EXEMPLO DE USO INTEGRADO
# =============================================================================

def complete_enhanced_recognition_system():
    """
    Sistema completo que combina várias técnicas para máxima acurácia.
    """

    print("=== SISTEMA DE RECONHECIMENTO FACIAL APRIMORADO ===")

    # 1. Cria galeria aprimorada
    print("\n1. Criando galeria com múltiplas referências...")
    enhanced_gallery = create_enhanced_gallery_multiple_references()

    # 2. Pré-processa imagem de teste
    print("\n2. Pré-processando imagem de teste...")
    probe_path = "/content/marcelinho_na_inferencia.jpg"
    processed_images = preprocess_image_for_better_recognition(probe_path, "/tmp/probe_processed.jpg")

    # 3. Reconhecimento com votação
    print("\n3. Executando reconhecimento com sistema de votação...")
    voting_results = enhanced_face_recognition_with_voting("/tmp/probe_processed.jpg", enhanced_gallery)

    # 4. Threshold adaptativo
    print("\n4. Calculando threshold adaptativo...")
    adaptive_result = adaptive_threshold_recognition("/tmp/probe_processed.jpg", enhanced_gallery)

    # 5. Apresenta resultados
    print("\n=== RESULTADOS ===")
    if voting_results:
        best_result = voting_results[0]
        print(f"Identidade: {best_result['identity']}")
        print(f"Distância mínima: {best_result['min_distance']:.4f}")
        print(f"Distância média: {best_result['avg_distance']:.4f}")
        print(f"Confiança: {best_result['confidence']:.2%}")
        print(f"Número de referências: {best_result['num_references']}")

    print(f"\nThreshold adaptativo: {adaptive_result['adaptive_threshold']:.4f}")
    print(f"Match válido: {adaptive_result['is_valid_match']}")

    return voting_results, adaptive_result

# Para executar o sistema completo:
# results = complete_enhanced_recognition_system()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.7/127.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.5 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=5812e707b4a3d5e7b10d65fcfdd7c85a081dfc3d90eb0ff669fd141321634639
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfu